# 17. Multi-Hypothesis Localization과 Kidnapped Robot Recovery

로봇 위치 belief가 항상 하나의 봉우리(unimodal)일 필요는 없다. 센서가 모호하거나 로봇이 납치(kidnapped)되면 belief는 여러 후보를 동시에 유지해야 한다.

$$bel(x_t)=\sum_k \alpha_k \mathcal{N}(x_t;\mu_k,\Sigma_k)$$

Particle filter는 이런 multi-modal belief를 자연스럽게 표현할 수 있다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False


## 1. 반복 패턴 지도에서의 모호한 관측

복도에 같은 관측 패턴이 반복되면, 한 번의 관측만으로는 위치가 여러 군데 가능하다.

In [ ]:
world = np.array([0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0])
n = len(world)
bel = np.ones(n) / n
p_hit = 0.88
p_false = 0.12

def correct(bel, z):
    likelihood = np.where(world == z, p_hit, p_false)
    out = bel * likelihood
    return out / out.sum()

def predict(bel, step=1):
    kernel = {-1: 0.08, 0: 0.16, 1: 0.76}
    out = np.zeros_like(bel)
    for i in range(n):
        for slip, p in kernel.items():
            out[(i + step + slip) % n] += bel[i] * p
    return out

history = [bel]
for z in [1, 0, 0, 1]:
    bel = correct(predict(bel), z)
    history.append(bel)

fig, axes = plt.subplots(len(history), 1, figsize=(8, 7), sharex=True)
for t, (ax, b) in enumerate(zip(axes, history)):
    ax.bar(np.arange(n), b, color='tab:purple')
    ax.set_ylim(0, max(0.45, b.max() * 1.15))
    ax.set_ylabel(f't={t}')
axes[-1].set_xlabel('cell')
axes[0].set_title('Multi-modal belief in a repetitive corridor')
plt.tight_layout()
plt.savefig('assets/17_multimodal_belief.png', dpi=160)
plt.show()

## 2. Kidnapped Robot 문제

Particle filter가 너무 한 곳에 수렴하면 갑작스러운 위치 변화에 약하다. 일부 random particle injection은 recovery를 도와준다.

In [ ]:
np.random.seed(17)
N = 700
particles = np.random.normal(3.0, 0.45, N) % n
true_before = 3
true_after = 9

def likelihood_at(x, z):
    cells = np.round(x).astype(int) % n
    return np.where(world[cells] == z, p_hit, p_false)

def resample(particles, weights):
    idx = np.random.choice(len(particles), size=len(particles), p=weights)
    return particles[idx]

snapshots = []
for t in range(6):
    true = true_before if t < 3 else true_after
    z = world[true]
    particles = (particles + np.random.normal(1.0, 0.35, N)) % n
    weights = likelihood_at(particles, z)
    weights = weights / weights.sum()
    particles = resample(particles, weights)
    if t >= 3:
        k = int(0.18 * N)
        particles[:k] = np.random.uniform(0, n, k)
    snapshots.append((t, true, particles.copy()))

fig, axes = plt.subplots(2, 3, figsize=(11, 5), sharex=True, sharey=True)
for ax, (t, true, ps) in zip(axes.ravel(), snapshots):
    ax.hist(ps, bins=np.arange(n + 1) - 0.5, density=True, color='tab:cyan', edgecolor='white')
    ax.axvline(true, color='tab:red', lw=2, label='true pose')
    ax.set_title(f't={t}, true={true}')
    ax.set_xlim(-0.5, n - 0.5)
axes[0,0].legend()
fig.suptitle('Random particle injection for kidnapped robot recovery')
plt.tight_layout()
plt.savefig('assets/17_kidnapped_robot_recovery.png', dpi=160)
plt.show()

## 3. 로보틱스 연결

| 개념 | 의미 | 로보틱스 활용 |
|---|---|---|
| Multi-modal belief | 여러 위치 가설을 동시에 유지 | global localization |
| Perceptual aliasing | 서로 다른 장소가 같은 관측을 생성 | 복도, 반복 구조 환경 |
| Kidnapped robot | 추정 위치와 실제 위치가 갑자기 달라짐 | robust localization 평가 |
| Random injection | 일부 particle을 전역에 재분산 | recovery 능력 향상 |

MCL이 EKF localization보다 강한 대표적인 이유가 바로 multi-modal belief 표현이다.